# Formal verification using CoHDL and Yosys

CoHDL is a hardware describtion language embedded in Python ([language documentation](https://github.com/alexander-forster/cohdl_documentation/tree/main/cohdl_introduction)).
[SymbiYosys](https://github.com/YosysHQ/sby) is a formal verification tool that supports VHDL and the Property Specification Language [PSL](https://en.wikipedia.org/wiki/Property_Specification_Language).

[cohdl_yosys](https://github.com/alexander-forster/cohdl_yosys) is a small Python library that adds formal verification support to CoHDL by invoking SymbiYosys.

cohdl_yosys consists of two parts:

* an abstraction layer that allows us to create and run SymbiYosys projects from Python
* a module of utility classes/functions that describe formal properties using Python syntax

In this documentation I describe the basic features of the library. I also try to give a brief introduction to formal verification in general.

Disclaimer:

I am by no means an expert on formal verification. This introduction is a condensed version of few weeks of experimenting with the tools.

Here are some resources I found particularly useful:

* https://zipcpu.com/blog/2017/10/19/formal-intro.html
* https://vhdlwhiz.com/formal-verification-in-vhdl-using-psl/
* https://en.wikipedia.org/wiki/Property_Specification_Language
* https://github.com/tmeissner/psl_with_ghdl

In [1]:
# basic setup of jupyter notebook

from __future__ import annotations

from example_util.jupyter_util import display_vcd

import cohdl

# By default, when an exception is raised during compilation,
# cohdl inserts fake stack frames into the traceback.
# This is useful for debugging but
# does not work properly inside jupyter notebooks.
cohdl.use_pretty_traceback(False)

## example description

As a first example we will verify the function of an (overly) simplistic temperature controller. The controller should have the following properties:

* monitor a 10 Bit sensor value
* the desired temperature corresponds to a sensor value of 500
* enable a heater if the temperature is to low
* enable a cooler if the temperature is to high
* heater and cooler are not allowed to run at the same time

We define the controller as a CoHDL entity. To make the example more interesting it contains a bug and violates the last requirement when the temperature is exactly 500.

In [2]:
from cohdl import std, Port, Bit, Unsigned

class TemperatureController(cohdl.Entity):

    clk = Port.input(Bit)

    temperature = Port.input(Unsigned[10])

    en_heater = Port.output(Bit, default=False)
    en_cooler = Port.output(Bit, default=False)

    def architecture(self):

        @std.sequential(std.Clock(self.clk))
        def ctrl_temperature():
            self.en_heater <<= self.temperature <= 500
            self.en_cooler <<= self.temperature >= 500


## testbenches

Before we dive into formal verification, lets first take a look at a traditional testbench.

The testbench generates some stimulus and applies it to the input of the tested unit. It then observes how the system reacts and checks that it matches the expected behaviour.

In [3]:
# use the ghdl_sim backend of cohdl_sim
# because it is the only one supported in Jupyter notebooks
from cohdl_sim.ghdl_sim import Simulator

sim = Simulator(TemperatureController, sim_args=["--vcd=build/waveform.vcd"])

# test is executed immediately when a function
# is declared with the sim.test decorator
@sim.test
async def testbench(dut: TemperatureController):
    sim.init_inputs()
    sim.gen_clock(dut.clk, std.MHz(100))

    for temp in (0, 100, 1000, 999, 333):
        dut.temperature <<= temp

        await sim.rising_edge(dut.clk)
        await sim.delta_step()

        assert dut.en_heater == Bit(temp <= 500)
        assert dut.en_cooler == Bit(temp >= 500)
        assert not (dut.en_heater and dut.en_cooler)

# display_vcd is a utility function that takes a vcd file, converts it to svg
# and displays it in the Jupyter notebook.

display_vcd(
    "build/waveform.vcd",
    "(*.)clk|temperature:d|en_heater|en_cooler",
    offset=1
)

build/waveform.vcd


loading VPI module '/opt/venv/lib/python3.12/site-packages/cohdl_sim_ghdl_interface.cpython-312-x86_64-linux-gnu.so'
VPI module loaded!


The test passes. We can also manually inspect the generated waveform and see that the design conforms to the specification for every tested timepoint. The testbench has failed to find the error because the relevant input value was not covered.

This is a fundamental problem because when writing the testbench, the tester only has three options:

1. run the test for all possible inputs

    This strategy ensures, that no edge-case was skipped. Unfortunately, the combinatorial explosion of input states make it impractical for everything but the simplest designs.

    If the example contained any memory elements, perhaps to produce a rolling average of the measurement, $2^{10*4}=1099511627776$ checks would be needed just to verify the functionality for 4 clock cycles.

2. select a subset of input values and check the requirements against them

    This is was we did in the example. Admittedly, the tester would probably have concluded from the requirement that 500 is a critical value and added it to the test set. Then again, nothing stops the implementer of the entity from introducing more subtle bugs. For example by defining the heater output like this: `self.en_heater <<= (self.temperature <= 500 or self.temperature == 678`.

3. review the implementation and write tests based on it

    If we can't test all possible input sequences and writing tests based on the requirement does not guard us against truly unexpected programmer errors, the only way for a tester to verify the functionality of a design is to review the implementation and adjust the test to it. In the example, the tester could conclude from the code that the output only depends on wheter the input is less-then, greater-than or equal-to 500 and loop over values that cover all these conditions.

    To some degree this process can be guided by fuzzing and code coverage tools but ultimately it is up to the tester to manually construct test sequences. If the implementation is changed, the testbench too needs to be reviewed.

## formal verification

Formal verification is essentially a way to automate the third approach. The tester does not look at the implementation and instead defines all requirements in a machine readable form. A verification tool then checks if they are fulfilled for all possible input values. Just like the human in the example, it does this by analyzing the tested code and using all sorts of logic optimizations to cut down the number of operations.

The rest of this document describes how assertions/assumptions and cover statements are used to describe formal properties of hardware designs.

### assertions

Like normal testbenches, formal verification tools support assertion statements to express that the tested design should (or shouldn't) have some property. `cohdl_yosys` does not use the Python `assert` statement directly. Instead it defines two callable objects `always` and `never`. When Yosys encounters such a statement it tries to find some input combination that violates the condition. 

```python
# essentially an assert statement that checks if condition is true
always["description"](condition)

# same as above but condition is inverted
never["description"](condition)
```

While an exhaustive search over all input states is not practical, we can still use it as a mental model for what we expect the tool to do. The `exhaustive_test` function loops over all possible inputs and checks that no assertion is violated. For now we will restrict the test to a limited number of clock cycles. This is called a bounded model check (BMC).

In [4]:
# pseudo code description of a bounded model check
def exhaustive_test(dut, depth):
    if depth == MAX_DEPTH:
        return
    
    for inp_state in all_inp_states:
        next_dut = dut.assign_inp(inp_state)

        if next_dut.any_assertion_fails():
            write_vcd_file_and_exit()
        
        exhaustive_test(next_dut, depth+1)

Now that we know a first type of formal statement, we can define our first test. We do this by creating a class that inherits from `YosysTestCase`. This class has two members. `_yosys_params_` to configure the Yosys project and an `architecture` method. The `architecture` serves the same purpose as in CoHDL entities. It contains synthesizable contexts (std.concurrent/std.sequential) that are transpiled from Python to VHDL. All CoHDL language constructs are supported. In addition we can use functions from the `cohdl_yosys.formal` module. These are convertet into [PSL](https://en.wikipedia.org/wiki/Property_Specification_Language) statements.

For now we just want to ensure that heater and cooler cannot run at the same time. We use a `never` statement to express the requirement and leave everything else to the tools.

In [5]:
from cohdl_yosys import YosysTestCase, YosysParams, formal
from cohdl_yosys.formal import never, always

class TemperatureController_Formal(YosysTestCase, entity=TemperatureController):
    _yosys_params_ = YosysParams(
        clean_build_dir=True, # clean yosys project directory before test
        quiet=True,           # suppress yosys output
        bmc=True,             # perform a bounded model check
    )

    def architecture(self, dut: TemperatureController):
        # yosys needs a reference clock
        formal.set_default_ctx(clk=std.Clock(dut.clk))

        # formal statements must be placed inside synthesizable contexts
        @std.concurrent
        def formal_properties():
            
            never["both_devices_running"](dut.en_cooler and dut.en_heater)


# The YosysTestCase class is derived from unittest.TestCase
# and can easily be integrated in an automated workflow.
# For this example we invoke the test function manually.
if TemperatureController_Formal().test_formal_properties(return_on_error=True):
    print("formal check has passed")
else:
    print("formal check has failed")
    display_vcd("build/**/*.vcd", "(*.)clk|temperature:d|en_heater|en_cooler")

formal check has failed
build/project_bmc/engine_0/trace.vcd


As expected the test has failed. Yosys has found a way to violate our assertion and gives us a .vcd waveform file that describes how. With that information we can fix the design and run the test again.

In [6]:
class TemperatureController(cohdl.Entity):

    clk = Port.input(Bit)

    temperature = Port.input(Unsigned[16])

    en_heater = Port.output(Bit, default=False)
    en_cooler = Port.output(Bit, default=False)

    def architecture(self):

        @std.sequential(std.Clock(self.clk))
        def ctrl_temperature():
            self.en_heater <<= self.temperature < 500
            self.en_cooler <<= self.temperature > 500

class TemperatureController_Formal(YosysTestCase, entity=TemperatureController):
    _yosys_params_ = YosysParams(
        clean_build_dir=True, bmc=True, quiet=True
    )

    def architecture(self, dut: TemperatureController):
        formal.set_default_ctx(clk=std.Clock(dut.clk))

        @std.concurrent
        def formal_properties():

            never["both_devices_running"](dut.en_cooler and dut.en_heater)

if TemperatureController_Formal().test_formal_properties(return_on_error=True):
    print("formal check has passed")
else:
    print("formal check has failed")
    display_vcd("build/**/*.vcd", "(*.)clk|temperature:d|en_heater|en_cooler")

formal check has passed


### assumptions

The next type of statement are assumptions. Assumptions allow us to exclude some input states from the test. At first this might sound strange. After all I just made the point that we should try to ensure the functionality for all inputs. But sometimes this is not practical, especially when dealing with bus clients. When all components of a system conform to the standard and only ever produce valid transactions, it would be a waste of resources to add error handling logic to every interface.

Rather than defining a list of allowed states directly, assumptions work like filters. The verification tool still tries all possible input values but will discard any that result in violated assumptions. 

We can add this logic to our simplified model by adding one if statement that continues with the next input state before assertions are verified.

In [7]:
def exhaustive_test(dut, depth):
    if depth == MAX_DEPTH:
        return
    
    for inp_state in all_inp_states:
        next_dut = dut.assign_inp(inp_state)

        if next_dut.any_assumption_fails():
            continue

        if next_dut.any_assertion_fails():
            write_vcd_file_and_exit()
        
        exhaustive_test(next_dut, depth+1)

To demonstrate assumptions, we will verify another entity. `SqrTable` returns the square of a given number by looking it up in a precomputed array.

In [8]:
class SqrTable(cohdl.Entity):

    clk = Port.input(Bit)

    idx = Port.input(Unsigned[16])
    value = Port.output(Unsigned[16], default=0)

    def architecture(self):
        buffer = std.Array[Unsigned[16], 16]([idx**2 for idx in range(16)])

        @std.concurrent
        def gen_value():
            self.value <<= buffer[self.idx]

Again, we describe the required properties of our design in a YosysTestCase. Again, the verification fails. This time because the tool found that addresses outside the range covered by the buffer produce wrong values.

In [9]:
class SqrTable_Formal(YosysTestCase, entity=SqrTable):
    _yosys_params_ = YosysParams(
        clean_build_dir=True, bmc=True, quiet=True
    )

    def architecture(self, dut: SqrTable):
        formal.set_default_ctx(clk=std.Clock(dut.clk))

        @std.concurrent
        def formal_properties():
            always["resut_is_sqr"](dut.value == dut.idx*dut.idx)

if SqrTable_Formal().test_formal_properties(return_on_error=True):
    print("formal check has passed")
else:
    print("formal check has failed")
    display_vcd("build/**/*.vcd", "(*.)clk|idx:d|value:d")

formal check has failed
build/project_bmc/engine_0/trace.vcd


There are ways to solve this, all of them have drawbacks:

1. increase the buffer size to cover all possible values

    Wastes much memory and would not even work here because the output type is not wide enough to represent all possible results.

2. change the input width of the entity to only allow valid inputs

    If we can freely choose the interface, this is the best solution. Not an option if the interface is outside our control like an existing bus system with a given address width.

3. add error output flag

    Possible, but now every user has to check for and handle an error that can never happen according to the specification.

If we can enforce that no user of the entity will ever pass an invalid input index, we can exclude such cases from the check.

In [10]:
from cohdl_yosys.formal import assume

class SqrTable_Formal(YosysTestCase, entity=SqrTable):
    _yosys_params_ = YosysParams(
        clean_build_dir=True, bmc=True, quiet=True
    )

    def architecture(self, dut: SqrTable):
        formal.set_default_ctx(clk=std.Clock(dut.clk))

        @std.concurrent
        def formal_properties():
            # only consider numbers that are less than 16
            assume["idx_less_than_16"](dut.idx < 16)

            always["resut_is_sqr"](dut.value == dut.idx*dut.idx)

if SqrTable_Formal().test_formal_properties(return_on_error=True):
    print("formal check has passed")
else:
    print("formal check has failed")
    display_vcd("build/**/*.vcd", "(*.)clk|idx:d|value:d")

formal check has passed


## cover statement

The cover statement is the last extension to our formal verification model. It marks a condition that we expect to be reachable. A coverage check fails when no sequence of input states exists that makes the condition true. If such a sequence exists, we get a .vcd file that describes it.

In [11]:
def exhaustive_test(dut, depth):
    if depth == MAX_DEPTH:
        if not all_cover_statements_reached():
            write_error_and_fail()
        
        return
    
    for inp_state in all_inp_states:
        next_dut = dut.assign_inp(inp_state)

        if next_dut.any_assumption_fails():
            continue

        if next_dut.any_assertion_fails():
            write_vcd_file_and_exit()
        
        if next_dut.cover_statement_reached():
            write_vcd_file()
        
        exhaustive_test(next_dut, depth+1)

Cover statements can also reveal issues with the testbench itself. For example when a wrong assumption restricts the search space to much.

In [12]:
from cohdl_yosys.formal import cover

class SqrTable_Formal(YosysTestCase, entity=SqrTable):
    _yosys_params_ = YosysParams(
        clean_build_dir=True, cover=True, quiet=True
    )

    def architecture(self, dut: SqrTable):
        formal.set_default_ctx(clk=std.Clock(dut.clk))

        @std.concurrent
        def formal_properties():
            # typo in assumption
            # yosys will only verify properties for index 0
            assume["idx_less_than_16"](dut.idx < 1)

            always["resut_is_sqr"](dut.value == dut.idx*dut.idx)
            
            # cover statement fails because
            # index 11 violates assumption
            cover["value_121"](dut.value == 121)

if SqrTable_Formal().test_formal_properties(return_on_error=True):
    print("formal check has passed")
    display_vcd("build/**/*.vcd", "(*.)clk|idx:d|value:d")
else:
    print("formal check has failed")

formal check has failed


When we fix the assumption and run the check again, the cover statement passes. The resulting waveform tells us that an index of 11 yields the result value 121.

In [13]:
from cohdl_yosys.formal import cover

class SqrTable_Formal(YosysTestCase, entity=SqrTable):
    _yosys_params_ = YosysParams(
        clean_build_dir=True, cover=True, quiet=True
    )

    def architecture(self, dut: SqrTable):
        formal.set_default_ctx(clk=std.Clock(dut.clk))

        @std.concurrent
        def formal_properties():
            assume["idx_less_than_16"](dut.idx < 16)

            always["resut_is_sqr"](dut.value == dut.idx*dut.idx)
            
            cover["value_121"](dut.value == 121)

if SqrTable_Formal().test_formal_properties(return_on_error=True):
    print("formal check has passed")
    display_vcd("build/**/*.vcd", "(*.)clk|idx:d|value:d")
else:
    print("formal check has failed")

formal check has passed
build/project_cover/engine_0/trace0.vcd


Figuring out that 11*11 equals 121 might not sound all that impressive but keep in mind that formal verification tools consider all possible paths. That makes the cover statement a very powerful tool to investigate the behaviour of a system. Even when there is no obvious way to reach some interesting target state.

The example below implements a simple CRC generator and then uses Yosys to find input sequences that lead to some desired output values.

In [14]:
class CrcGenerator(cohdl.Entity):

    clk = Port.input(Bit)

    data_in = Port.input(Bit)
    data_out = Port.output(Unsigned[8], default=0)

    def architecture(self):
        # a crc generator with some randomly choosen polynomial
        crc = std.crc.BitwiseCrc(std.as_bitvector("01101001"))

        @std.sequential(std.Clock(self.clk))
        def calc_crc():
            crc.update(self.data_in)
            self.data_out <<= crc.result()

In [15]:
class CrcReverse(YosysTestCase, entity=CrcGenerator):
    _yosys_params_ = YosysParams(
        clean_build_dir=True, cover=True, quiet=True
    )

    def architecture(self, dut: CrcGenerator):
        formal.set_default_ctx(clk=std.Clock(dut.clk))

        @std.concurrent
        def formal_properties():

            cover["out_00"](dut.data_out == 0x00)
            cover["out_AB"](dut.data_out == 0xAB)
            cover["out_FF"](dut.data_out == 0xFF)

if CrcReverse().test_formal_properties(return_on_error=True):
    print("formal check has passed")
    display_vcd("build/**/*.vcd", "(*.)clk|data_in|data_out")

formal check has passed
build/project_cover/engine_0/trace0.vcd


build/project_cover/engine_0/trace1.vcd


build/project_cover/engine_0/trace2.vcd


## labels of formal statements

The label passed to always/never/assume/cover-statements is directly injected into the generated VHDL representation. cohdl_yosys does currently not sanitize this. If the string does not conform to the VHDL/PSL identifier rules, Yosys will reject the test.

The description labels provide are useful to interpret the output of Yosys. Functionally, they are not required and therefore optional.

```python

# label string must follow VHDL/PSL identifier rules
always["some_condition"](some_condition)

# use an empty slice to create a statement without label
always[:](some_condition)

# the second slice element is ignored, can be used as a comment
always[:"this label is only visible in Python and can thus be an arbitrary string!"](some_condition)
```